# Arm SDK low-level joint control

This notebook creates a Jupyter joint-control UI for `rt/arm_sdk`. Participants select an arm joint, nudge or set its target pose, and the publisher ramps every command incrementally at a configured speed.

`rt/arm_sdk` controls the upper body only and does not require developer mode, but it still sends real servo targets. Sync to the live state before moving.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import DDS, widgets, and the same arm joint constants used by the local slider app.


In [ ]:
import threading
import time
from dataclasses import dataclass
from typing import Any

import ipywidgets as widgets
from IPython.display import display

from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

LEFT_ARM_IDX = [15, 16, 17, 18, 19, 20, 21]
RIGHT_ARM_IDX = [22, 23, 24, 25, 26, 27, 28]
NOT_USED_IDX = 29
JOINT_NAMES = ["shoulder_pitch", "shoulder_roll", "shoulder_yaw", "elbow", "wrist_pitch", "wrist_roll", "wrist_yaw"]
ALL_ARM_JOINTS = LEFT_ARM_IDX + RIGHT_ARM_IDX


Define a lowstate reader and a ramping publisher. The publish loop limits joint target changes by `speed_rad_s / rate_hz` on every tick.


In [ ]:
def resolve_lowstate_type():
    for module_path in ("unitree_sdk2py.idl.unitree_hg.msg.dds_", "unitree_sdk2py.idl.unitree_go.msg.dds_"):
        try:
            module = __import__(module_path, fromlist=["LowState_"])
            return getattr(module, "LowState_")
        except Exception:
            continue
    raise RuntimeError("LowState_ type not found.")


class ArmState:
    def __init__(self, joints):
        self.joints = list(joints)
        self._lock = threading.Lock()
        self.positions = {}
        self.ts = 0.0
        self.sub = ChannelSubscriber("rt/lowstate", resolve_lowstate_type())
        self.sub.Init(self._callback, 200)

    def _callback(self, msg: Any):
        try:
            data = {joint: float(msg.motor_state[joint].q) for joint in self.joints}
        except Exception:
            return
        with self._lock:
            self.positions = data
            self.ts = time.time()

    def snapshot(self):
        with self._lock:
            return dict(self.positions), self.ts

    def wait(self, timeout=3.0):
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            data, ts = self.snapshot()
            if data:
                return data
            time.sleep(0.02)
        raise TimeoutError("Timed out waiting for rt/lowstate arm data.")


class ArmRampController:
    def __init__(self, iface, domain_id, joints):
        ChannelFactoryInitialize(int(domain_id), str(iface))
        self.joints = list(joints)
        self.state = ArmState(self.joints)
        self.pub = ChannelPublisher("rt/arm_sdk", LowCmd_)
        self.pub.Init()
        self.cmd = unitree_hg_msg_dds__LowCmd_()
        self.cmd.motor_cmd[NOT_USED_IDX].q = 1.0
        self.crc = CRC()
        self.kp = 30.0
        self.kd = 1.5
        self.speed_rad_s = 0.35
        self.rate_hz = 50.0
        self.current_targets = {joint: 0.0 for joint in self.joints}
        self.desired_targets = dict(self.current_targets)
        self.running = False
        self._lock = threading.RLock()
        self._stop = threading.Event()
        self._thread = None

    def sync(self):
        data = self.state.wait()
        with self._lock:
            self.current_targets = dict(data)
            self.desired_targets = dict(data)
        return "Targets synced to current arm state."

    def set_target(self, joint, q):
        with self._lock:
            self.desired_targets[int(joint)] = float(q)
        return f"joint {joint} target set to {float(q):+.3f} rad"

    def nudge(self, joint, delta):
        with self._lock:
            base = self.desired_targets.get(int(joint), self.current_targets.get(int(joint), 0.0))
            self.desired_targets[int(joint)] = float(base) + float(delta)
            return f"joint {joint} target nudged to {self.desired_targets[int(joint)]:+.3f} rad"

    def start(self):
        if not self.current_targets or all(v == 0.0 for v in self.current_targets.values()):
            self.sync()
        self.running = True
        if self._thread is None or not self._thread.is_alive():
            self._stop.clear()
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        return "Ramping publisher running."

    def stop(self):
        self.running = False
        return "Ramping publisher paused; last command is held by the robot controller."

    def zero_gains_once(self):
        hold = self.state.wait()
        for joint in self.joints:
            mc = self.cmd.motor_cmd[joint]
            mc.mode = 1
            mc.q = float(hold[joint])
            mc.dq = 0.0
            mc.kp = 0.0
            mc.kd = 0.0
            mc.tau = 0.0
        self.cmd.crc = self.crc.Crc(self.cmd)
        self.pub.Write(self.cmd)
        return "Published one zero-gain arm_sdk packet."

    def _loop(self):
        dt = 1.0 / max(1.0, self.rate_hz)
        while not self._stop.is_set():
            if self.running:
                with self._lock:
                    max_step = max(0.001, self.speed_rad_s * dt)
                    for joint in self.joints:
                        cur = self.current_targets[joint]
                        des = self.desired_targets[joint]
                        step = max(-max_step, min(max_step, des - cur))
                        self.current_targets[joint] = cur + step
                    self._publish_locked()
            time.sleep(dt)

    def _publish_locked(self):
        for joint in self.joints:
            mc = self.cmd.motor_cmd[joint]
            mc.mode = 1
            mc.q = float(self.current_targets[joint])
            mc.dq = 0.0
            mc.kp = float(self.kp)
            mc.kd = float(self.kd)
            mc.tau = 0.0
        self.cmd.crc = self.crc.Crc(self.cmd)
        self.pub.Write(self.cmd)

arm_control = ArmRampController(IFACE, DOMAIN_ID, ALL_ARM_JOINTS)
print("Arm ramp controller ready.")


Run the UI. Select one joint, sync to the live state, start the ramping publisher, and then make small changes.


In [ ]:
options = []
for side, joints in (("left", LEFT_ARM_IDX), ("right", RIGHT_ARM_IDX)):
    for offset, joint in enumerate(joints):
        options.append((f"{side} {JOINT_NAMES[offset]} ({joint})", joint))
joint = widgets.Dropdown(options=options, description="Joint")
target = widgets.FloatSlider(value=0.0, min=-3.14, max=3.14, step=0.01, description="Target rad", readout_format=".2f", layout=widgets.Layout(width="520px"))
nudge = widgets.FloatSlider(value=0.05, min=0.005, max=0.25, step=0.005, description="Nudge")
speed = widgets.FloatSlider(value=0.35, min=0.02, max=1.5, step=0.02, description="Speed")
kp = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0, description="kp")
kd = widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description="kd")
sync = widgets.Button(description="Sync", button_style="info")
start = widgets.Button(description="Start Ramp", button_style="success")
stop = widgets.Button(description="Pause", button_style="warning")
minus = widgets.Button(description="- Nudge")
plus = widgets.Button(description="+ Nudge")
zero = widgets.Button(description="Zero Gains", button_style="danger")
status = widgets.HTML(value="")


def refresh_target_from_desired(*_):
    data, ts = arm_control.state.snapshot()
    selected = int(joint.value)
    if selected in arm_control.desired_targets:
        target.value = float(arm_control.desired_targets[selected])
    status.value = f"selected joint={selected} live={data.get(selected, None)} lowstate_age={time.time() - ts:.2f}s" if ts else "Waiting for lowstate."


def apply_gains(*_):
    arm_control.speed_rad_s = float(speed.value)
    arm_control.kp = float(kp.value)
    arm_control.kd = float(kd.value)


def on_target(change):
    if change["name"] == "value":
        apply_gains()
        status.value = arm_control.set_target(joint.value, change["new"])

sync.on_click(lambda _: (setattr(status, "value", arm_control.sync()), refresh_target_from_desired()))
start.on_click(lambda _: (apply_gains(), setattr(status, "value", arm_control.start())))
stop.on_click(lambda _: setattr(status, "value", arm_control.stop()))
minus.on_click(lambda _: (setattr(status, "value", arm_control.nudge(joint.value, -nudge.value)), refresh_target_from_desired()))
plus.on_click(lambda _: (setattr(status, "value", arm_control.nudge(joint.value, nudge.value)), refresh_target_from_desired()))
zero.on_click(lambda _: setattr(status, "value", arm_control.zero_gains_once()))
joint.observe(lambda change: refresh_target_from_desired(), names="value")
target.observe(on_target, names="value")
for w in (speed, kp, kd):
    w.observe(lambda change: apply_gains(), names="value")
refresh_target_from_desired()
display(widgets.VBox([widgets.HBox([joint, target]), widgets.HBox([nudge, speed]), widgets.HBox([kp, kd]), widgets.HBox([sync, start, stop, minus, plus, zero]), status]))
